# build_tables.ipynb
Scrapes WPI course listings, then builds the program and requirement tables from `program_data/*.yaml`. Everything is written to `tables/`.

## 1. Courses

Imports and a pattern that splits `"CS 1101 - Intro"` into code and title.

In [11]:
import re

import pandas as pd
import requests

# "CS 1101 - Introduction to Program Design" -> code, title
TITLE_RE = re.compile(r"^(?P<code>[A-Za-z]+ ?\d+[A-Za-z]*) - (?P<title>.*)$")

Download every course section from the WPI course listings.

In [12]:
# one entry per section, not per course
entries = requests.get("https://courselistings.wpi.edu/assets/prod-data.json", timeout=30).json()["Report_Entry"]
print(len(entries), "sections")

3771 sections


Turn sections into one row per course (latest section wins), with credits converted between undergrad and grad.

In [13]:
rows = []
for e in entries:
    m = TITLE_RE.match(e["Course_Title"])
    credits = float(e["Credits"])
    # 1 grad credit = 1.5 undergrad credits
    grad = e["Academic_Level"] == "Graduate"
    rows.append(
        {
            "course_code": m["code"],
            "title": m["title"],
            # strip HTML tags
            "description": re.sub(r"<[^>]+>", "", e["Course_Description"]),
            "subject": e["Subject"],
            "credits_raw": credits,
            "credits_ug": credits * 3 / 2 if grad else credits,
            "credits_gr": credits if grad else credits * 2 / 3,
            "academic_level": e["Academic_Level"],
            "department": e["Course_Section_Owner"],
            "start_date": e["Course_Section_Start_Date"],
        }
    )

# one row per course, keeping the latest section
courses_df = pd.DataFrame(rows).sort_values("start_date", kind="stable").drop_duplicates("course_code", keep="last")
print(len(courses_df), "courses")
courses_df.head()

1281 courses


,course_code,title,description,subject,credits_raw,credits_ug,credits_gr,academic_level,department,start_date
0,AB 1531,Elementary Arabic I,Cat. IAn intensive course to introduce the Ara...,Arabic,3.0,3.0,2.0,Undergraduate,Humanities and Arts Department,2026-08-20
43,AE 3110,Fundamentals of Compressible Fluid Dynamics,"In this course, students are introduced to var...",Aerospace Engineering,3.0,3.0,2.0,Undergraduate,Aerospace Engineering Department,2026-08-20
45,AE 3310,Fundamentals of Navigation and Communication,This course covers methods and current technol...,Aerospace Engineering,3.0,3.0,2.0,Undergraduate,Aerospace Engineering Department,2026-08-20
48,AE 4210,Fundamentals of Air-Breathing Propulsion,This course introduces the principles of opera...,Aerospace Engineering,3.0,3.0,2.0,Undergraduate,Aerospace Engineering Department,2026-08-20
49,AE 4220,Fundamentals of Rocket Propulsion,This course provides a study of rocket propuls...,Aerospace Engineering,3.0,3.0,2.0,Undergraduate,Aerospace Engineering Department,2026-08-20


Save `courses_undergrad.csv`, `courses_grad.csv` and `departments.csv`.

In [14]:
# split on academic_level
courses_df[courses_df["academic_level"] == "Undergraduate"].drop(columns="credits_ug").to_csv("tables/courses_undergrad.csv", index=False)
courses_df[courses_df["academic_level"] == "Graduate"].drop(columns="credits_gr").to_csv("tables/courses_grad.csv", index=False)
# one row per department
courses_df[["department"]].drop_duplicates().sort_values("department").rename(columns={"department": "dept_name"}).to_csv("tables/departments.csv", index=False)

## 2. Programs and requirements

Load the courses and every program yaml. The header comment lists the fields a requirement can have.

In [15]:
# Turns program_data/*.yaml into CSVs.
# Requirement fields (only name is required):
#   min_credits             credits needed
#   max_credits             most credits that can count
#   min_children            children needed (default: all)
#   min_courses             course enrollments needed (repeats count, e.g. 0 cr seminars)
#   courses                 courses that count
#   match                   rule that finds more courses in the course tables
# A parent's credits include its children's courses.

import glob
import re

import pandas as pd
import yaml

# all offered courses (both levels)
courses_df = pd.concat([pd.read_csv("tables/courses_undergrad.csv"), pd.read_csv("tables/courses_grad.csv")], ignore_index=True)
# one per program
programs = [yaml.safe_load(open(f)) for f in sorted(glob.glob("program_data/*.yaml"))]

`slug` makes a URL-style id from a requirement name.

In [16]:
def slug(name):
    # "CS Breadth" -> "cs-breadth"
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")

`expand_match` finds the courses in `courses.csv` that fit a `match` rule.

In [17]:
def expand_match(match, df):
    # start with all courses, then filter by each rule
    sel = pd.Series(True, index=df.index)
    if "subject_contains" in match:
        # by subject name, so cross-listed courses count
        sel &= df["subject"].str.contains(match["subject_contains"], case=False, na=False)
    if "academic_level" in match:
        # grad or undergrad
        sel &= df["academic_level"] == match["academic_level"]
    if "code_prefix" in match:
        # e.g. "RBE" or [MIS, OIE]
        prefixes = match["code_prefix"]
        sel &= df["course_code"].str.split().str[0].isin([prefixes] if isinstance(prefixes, str) else prefixes)
    if "code_min" in match:
        # "CS 411X" -> 411
        sel &= pd.to_numeric(df["course_code"].str.extract(r"(\d+)")[0], errors="coerce") >= match["code_min"]
    if "code_max" in match:
        sel &= pd.to_numeric(df["course_code"].str.extract(r"(\d+)")[0], errors="coerce") <= match["code_max"]
    # drop excluded courses
    return sorted(set(df.loc[sel, "course_code"]) - set(match.get("exclude", [])))

`walk` records one requirement, its courses, and then its children (recursive).

In [18]:
# rows for the three tables
program_rows, requirement_rows, link_rows = [], [], []


def walk(node, program_id, parent_id, req_id):
    # save this requirement, then its children
    # id is a path, e.g. "cs-bsms/cs-breadth/theory"
    children = node.get("children", [])
    requirement_rows.append(
        {
            "id": req_id,
            "path": req_id[len(program_id) + 1 :],
            "program_id": program_id,
            "parent_id": parent_id,
            "name": node["name"],
            "min_credits": node.get("min_credits"),
            "max_credits": node.get("max_credits"),
            # default: all children
            "min_children": node.get("min_children", len(children) or None),
            "min_courses": node.get("min_courses"),
        }
    )
    # listed courses + matched courses
    codes = node.get("courses", [])
    if "match" in node:
        codes = codes + expand_match(node["match"], courses_df)
    link_rows.extend({"requirement_id": req_id, "course_code": c} for c in codes)
    for child in children:
        walk(child, program_id, req_id, f"{req_id}/{slug(child['name'])}")

Walk every program's requirements and collect the rows into dataframes.

In [19]:
for p in programs:
    program_rows.append({k: p.get(k) for k in ["id", "name", "school", "catalog_url", "total_credits"]})
    for req in p["requirements"]:
        # top level has no parent
        walk(req, p["id"], None, f"{p['id']}/{slug(req['name'])}")

programs_df = pd.DataFrame(program_rows)
requirements_df = pd.DataFrame(requirement_rows)
# remove duplicate links
links_df = pd.DataFrame(link_rows).drop_duplicates()

Write `programs.csv`, `schools.csv`, `requirements.csv` and `filledby.csv`.

In [20]:
# write the CSVs
programs_df.to_csv("tables/programs.csv", index=False)
# one row per school
programs_df[["school"]].drop_duplicates().sort_values("school").rename(columns={"school": "name"}).to_csv("tables/schools.csv", index=False)
# blanks become 0 so the import has no empty numeric cells
int_cols = ["min_credits", "max_credits", "min_children", "min_courses"]
requirements_df[int_cols] = requirements_df[int_cols].fillna(0).astype(int)
requirements_df.to_csv("tables/requirements.csv", index=False)
links_df.to_csv("tables/filledby.csv", index=False)